# Population Dynamics: The Leslie Matrix Model
## Capstone Project Notebook

Imagine you are studying an animal population. You know that birth rates and survival rates often depend heavily on an individual's age. Young individuals might not reproduce yet, prime-age adults might have high birth rates, and older individuals might have lower survival rates. How can you model the *entire* population's growth and structure over time, taking these age differences into account?

This is where the **Leslie Matrix model** comes in. Developed by Patrick H. Leslie in the 1940s, it is a powerful tool in population ecology used to model the growth of populations that are structured by **age classes**. It operates in discrete time steps (e.g., year to year) and uses linear algebra (specifically matrix multiplication) to project the future size and age distribution of a population.

At its core, the Leslie Matrix model relies on two key components:
- **Fertility rates** — the average number of offspring produced by individuals in each age class
- **Survival probabilities** — the proportion of individuals surviving from one age class to the next

These values are arranged into a square matrix (the Leslie matrix), which captures how individuals move through age classes over time and how many new individuals are added. By multiplying this matrix with a population vector (containing the number of individuals in each age class at a given time), we obtain the projected population vector for the next time step.

### Why Use the Leslie Matrix?

- **Age matters:** It explicitly incorporates age-specific birth (fecundity) and survival rates.
- **Predictive power:** It allows you to predict the population size and structure in future time steps.
- **Long-term behaviour:** The dominant eigenvalue of the Leslie matrix corresponds to the long-term growth rate, while the associated eigenvector gives the **stable age distribution** — the proportions of individuals in each age class that the population tends towards over time.
- **Management insights:** Useful in conservation biology, wildlife management, fisheries, and human demography to assess how changes in survival or fertility affect population dynamics and to identify critical age classes for targeted intervention.

### Comparison with Simpler Models

The **logistic growth model** $N_{t+1} = N_t + rN_t(1 - N_t/K)$ captures density-dependent growth but treats the population as homogeneous — every individual contributes equally to reproduction and survival regardless of age. The Leslie Matrix model incorporates **demographic structure**, recognising that individuals of different ages contribute differently. Rather than modelling a single population size variable, it tracks a vector of age-specific population counts:

$$\mathbf{n}_{t+1} = \mathbf{L}\, \mathbf{n}_t$$

This allows detailed modelling of phenomena such as delayed reproduction, age-specific mortality, population momentum, and long-term age structure convergence.

### In this notebook you will:
1. Understand **age-structured population dynamics** and why age matters
2. Build a **Leslie matrix** from fertility and survival rates
3. Project population forward in time using **matrix multiplication**
4. Analyse **eigenvalues** to find long-term growth rate and stable age distribution
5. Visualise population pyramids and demographic transitions
6. Lay the groundwork for required project extensions

---

## 0 · Setup

We import NumPy for matrix operations and Matplotlib for plotting. The Leslie matrix model relies heavily on linear algebra — matrix multiplication for population projection and eigenvalue decomposition for long-term analysis — both provided by NumPy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
})

---
## 1 · Why Age Structure Matters

Simple models (Malthusian, logistic) treat the entire population as homogeneous. In reality:

- **Different age groups reproduce at different rates** — teenagers and elderly have very different fertility.
- **Survival rates vary by age** — infant mortality differs from adult mortality.
- **Population momentum** — a young population continues growing even after fertility drops, because many individuals haven't yet reached reproductive age.

The **Leslie matrix** (Patrick H. Leslie, 1945) captures these effects by tracking the number of individuals in each age class separately.

### The Core Components

**Age Classes:** The population is divided into a set number of discrete age classes (e.g., 0–1 year, 1–2 years, 2–3 years, etc.). Let's say there are $k$ age classes.

**Population Vector:** A column vector representing the number of individuals in each age class at a specific time $t$:

$$\mathbf{N}_t = \begin{bmatrix} n_{1,t} \\ n_{2,t} \\ \vdots \\ n_{k,t} \end{bmatrix}$$

**Leslie Matrix ($\mathbf{L}$):** A square ($k \times k$) matrix containing the demographic rates:
- **Fecundity rates ($f_i$):** The average number of newborn offspring produced by an individual in age class $i$ during one time step, that survive to enter the first age class at the next time step. These form the **first row**.
- **Survival rates ($s_i$):** The probability that an individual in age class $i$ survives to enter age class $i+1$ in the next time step. These form the **sub-diagonal**.
- All other entries are typically zero.

### The Projection Equation

To find the population vector at the next time step, multiply the Leslie matrix by the current population vector:

$$\mathbf{N}_{t+1} = \mathbf{L} \times \mathbf{N}_t$$

This equation calculates:
- The number of **newborns** in the next time step (sum of $f_i \times n_{i,t}$ for all $i$)
- The number of **survivors** moving into each subsequent age class ($s_i \times n_{i,t}$ gives the number moving from class $i$ to $i+1$)

If $\lambda_1 > 1$, the population grows; if $\lambda_1 < 1$, it shrinks; if $\lambda_1 = 1$, it is stationary in the long term.

---
## 2 · The Leslie Matrix

Consider a population divided into $n$ age classes, each spanning a fixed time interval (e.g., 5 years). Define:

- $\mathbf{N}(t) = [N_1(t), N_2(t), \ldots, N_n(t)]^T$ — population vector at time $t$
- $f_i$ — **fertility rate** of age class $i$ (average offspring per individual per time step)
- $s_i$ — **survival rate** from age class $i$ to $i+1$ (probability of surviving to the next class)

The Leslie matrix has the form:

$$\mathbf{L} = \begin{pmatrix}
f_1 & f_2 & f_3 & \cdots & f_n \\
s_1 & 0   & 0   & \cdots & 0   \\
0   & s_2 & 0   & \cdots & 0   \\
\vdots & & \ddots & & \vdots \\
0   & 0   & \cdots & s_{n-1} & 0
\end{pmatrix}$$

Population update: $\mathbf{N}(t+1) = \mathbf{L} \cdot \mathbf{N}(t)$

- **First row**: new births from each age class
- **Sub-diagonal**: survivors advancing to the next age class

### Step 1: Build the Leslie matrix

The function `build_leslie_matrix` takes two arrays — fertility rates (length $n$) and survival rates (length $n-1$) — and assembles them into the standard Leslie matrix structure. Fertility rates fill the first row, and survival rates fill the sub-diagonal. All other entries are zero.

We test it with a simple 3-class example: juveniles (class 1, no reproduction), prime adults (class 2, fertility 1.6), and old adults (class 3, fertility 0.8). Survival from class 1→2 is 0.5 (50%), and from class 2→3 is 0.7 (70%).

In [ ]:
def build_leslie_matrix(fertility, survival):
    """Construct a Leslie matrix from fertility and survival arrays.
    
    Parameters
    ----------
    fertility : array of length n — fertility rates for each age class
    survival  : array of length n-1 — survival rates between consecutive classes
    
    Returns
    -------
    L : n x n Leslie matrix
    """
    n = len(fertility)
    L = np.zeros((n, n))
    L[0, :] = fertility
    for i in range(n - 1):
        L[i + 1, i] = survival[i]
    return L

# Build a simple 3-class example
f_test = np.array([0.0, 1.6, 0.8])
s_test = np.array([0.5, 0.7])
L_test = build_leslie_matrix(f_test, s_test)

### Step 2: One-step population update

The beauty of the Leslie model is that a single matrix multiplication computes the entire next generation. The operation $\mathbf{N}_1 = \mathbf{L} \cdot \mathbf{N}_0$ simultaneously:
- Sums up the births from all age classes (first row of $\mathbf{L}$ dotted with $\mathbf{N}_0$)
- Advances survivors from each class to the next (sub-diagonal entries)

Below we verify this by manually checking the arithmetic: new births = $0 \times 10 + 1.6 \times 8 + 0.8 \times 5 = 16.8$, survivors from class 1 = $0.5 \times 10 = 5.0$, survivors from class 2 = $0.7 \times 8 = 5.6$.

In [ ]:
N0 = np.array([10.0, 8.0, 5.0])  # initial population by age class
N1 = L_test @ N0

print(f'Initial: {N0}   total = {N0.sum():.0f}')
print(f'After 1 step: {N1}   total = {N1.sum():.0f}')
print(f'\nNew births: {f_test @ N0:.1f} (= 0*10 + 1.6*8 + 0.8*5)')
print(f'Survivors from class 1→2: {s_test[0]*N0[0]:.1f}')
print(f'Survivors from class 2→3: {s_test[1]*N0[1]:.1f}')

---
## 3 · Multi-Step Simulation

To project the population over many time steps, we simply apply the matrix multiplication repeatedly: $\mathbf{N}_t = \mathbf{L}^t \cdot \mathbf{N}_0$. The function `simulate_leslie` does this iteratively and stores the full history as a 2D array (age classes × time steps).

We visualise two aspects:
- **Total population** (left) — shows overall growth or decline. After an initial transient, growth becomes approximately geometric (exponential in continuous time).
- **Age structure** (right, stacked area plot) — shows how the proportion of each age class changes over time. Initially the proportions fluctuate, but they eventually stabilise at the **stable age distribution** predicted by the dominant eigenvector.

In [ ]:
def simulate_leslie(L, N0, num_steps):
    """Simulate population dynamics using the Leslie matrix.
    
    Returns
    -------
    history : (n_classes, num_steps+1) array of population counts
    """
    n = len(N0)
    history = np.zeros((n, num_steps + 1))
    history[:, 0] = N0
    Nt = N0.copy()
    for t in range(num_steps):
        Nt = L @ Nt
        history[:, t + 1] = Nt
    return history

# Run 20-step simulation
history = simulate_leslie(L_test, N0, 20)
total = np.sum(history, axis=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: total population
ax1.plot(total, 'b-o', lw=2, markersize=4)
ax1.set_xlabel('Time Step')
ax1.set_ylabel('Total Population')
ax1.set_title('Total Population Growth', fontweight='bold')
ax1.grid(True, alpha=0.3)

# Right: age structure over time (stacked)
labels = [f'Age Class {i+1}' for i in range(len(N0))]
age_fracs = history / total[np.newaxis, :]
ax2.stackplot(range(history.shape[1]), *age_fracs, labels=labels, alpha=0.8)
ax2.set_xlabel('Time Step')
ax2.set_ylabel('Proportion')
ax2.set_title('Age Structure Over Time', fontweight='bold')
ax2.legend(loc='upper right', fontsize=9)
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.show()

---
## 4 · Eigenvalue Analysis

The **dominant eigenvalue** $\lambda_1$ of $\mathbf{L}$ gives the long-term growth factor:
- $\lambda_1 > 1$: population grows
- $\lambda_1 = 1$: population is stable
- $\lambda_1 < 1$: population declines

The corresponding **eigenvector** gives the **stable age distribution** — the proportions each age class converges to regardless of initial conditions.

The function `analyse_leslie` computes the full eigendecomposition using `np.linalg.eig`, identifies the dominant (largest real) eigenvalue, and normalises the corresponding eigenvector to sum to 1 (so it represents proportions). We then compare this theoretical stable distribution with the simulated age fractions at $t = 20$ — they should be nearly identical, confirming convergence.

In [ ]:
def analyse_leslie(L):
    """Compute dominant eigenvalue and stable age distribution."""
    eigenvalues, eigenvectors = np.linalg.eig(L)
    idx = np.argmax(np.real(eigenvalues))
    lambda_dom = np.real(eigenvalues[idx])
    v_dom = np.real(eigenvectors[:, idx])
    stable_dist = v_dom / np.sum(v_dom)
    return lambda_dom, stable_dist

lam, stable = analyse_leslie(L_test)

### Convergence to stable age distribution

A remarkable property of the Leslie model is that the stable age distribution is an **attractor** — the population converges to it from *any* initial condition, as long as the Leslie matrix is primitive (which requires at least two consecutive age classes with positive fertility and all survival rates positive).

Below we demonstrate this by starting from three extreme initial conditions: all individuals in the youngest class, all in the oldest class, and a uniform distribution. Despite these very different starting points, all three converge to the same stable proportions (shown as dashed lines) within about 10–15 time steps. This convergence is a direct consequence of the **Perron-Frobenius theorem** from linear algebra.

In [ ]:
# Try different initial conditions — all converge to the same stable distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
initial_conditions = [
    np.array([100.0, 0.0, 0.0]),
    np.array([0.0, 0.0, 100.0]),
    np.array([33.0, 34.0, 33.0]),
]
titles = ['All Young', 'All Old', 'Uniform']

for ax, N0_ic, title in zip(axes, initial_conditions, titles):
    hist = simulate_leslie(L_test, N0_ic, 20)
    tot = np.sum(hist, axis=0)
    fracs = hist / tot[np.newaxis, :]
    for i in range(len(N0_ic)):
        ax.plot(fracs[i], lw=2, label=f'Class {i+1}')
    for i, s in enumerate(stable):
        ax.axhline(s, color=f'C{i}', ls='--', lw=0.8, alpha=0.5)
    ax.set_title(f'IC: {title}', fontweight='bold')
    ax.set_xlabel('Time'); ax.set_ylabel('Fraction')
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)

fig.suptitle('Convergence to Stable Age Distribution (dashed lines)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

---
## 5 · A More Realistic Example

The 3-class toy model above demonstrates the mechanics. Now let's apply the Leslie matrix to a more realistic scenario: an 11-class model with 5-year age groups spanning 0–50+ years, using plausible demographic parameters for a developing country's **female population**.

Key features of these parameters:
- **Zero fertility** for ages 0–9 (pre-reproductive) and 50+ (post-reproductive)
- **Peak fertility** at ages 25–29 (fertility = 0.300)
- **High survival** across all classes (>97%), with slightly lower survival for the youngest (infant/child mortality) and oldest groups
- **Pyramidal initial population** — more individuals in younger classes, tapering with age

We project this population 200 years into the future (40 steps × 5 years) and display population pyramids at Year 0 and Year 100 to show how the age structure evolves.

In [ ]:
fertility_real = np.array([0.000, 0.000, 0.005, 0.100, 0.250,
                           0.300, 0.250, 0.100, 0.020, 0.002, 0.000])
survival_real  = np.array([0.985, 0.997, 0.998, 0.997, 0.996,
                           0.995, 0.993, 0.990, 0.985, 0.970])

L_real = build_leslie_matrix(fertility_real, survival_real)

# Initial population (thousands) — approximate a developing country
N0_real = np.array([500, 480, 460, 440, 400, 350, 300, 250, 200, 150, 100], dtype=float)

history_real = simulate_leslie(L_real, N0_real, 40)  # 40 steps × 5 years = 200 years

lam_real, stable_real = analyse_leslie(L_real)

# Population pyramid at t=0 and t=20 (100 years)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
age_labels = ['0-4', '5-9', '10-14', '15-19', '20-24', '25-29',
              '30-34', '35-39', '40-44', '45-49', '50+']
y_pos = np.arange(len(age_labels))

ax1.barh(y_pos, history_real[:, 0], color='steelblue', height=0.6)
ax1.set_yticks(y_pos); ax1.set_yticklabels(age_labels)
ax1.set_xlabel('Population (thousands)')
ax1.set_title('Population Pyramid — Year 0', fontweight='bold')
ax1.invert_yaxis()

ax2.barh(y_pos, history_real[:, 20], color='coral', height=0.6)
ax2.set_yticks(y_pos); ax2.set_yticklabels(age_labels)
ax2.set_xlabel('Population (thousands)')
ax2.set_title('Population Pyramid — Year 100', fontweight='bold')
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

---
## 6 · Total Population and Growth Trajectory

Two diagnostic plots reveal the long-term dynamics:

- **Total population** (left) — shows the overall trajectory. After an initial transient (where the age structure adjusts from the arbitrary initial condition toward the stable distribution), growth becomes approximately geometric with factor $\lambda_1$ per time step.
- **Growth factor per step** (right) — the ratio $N_\text{total}(t+1) / N_\text{total}(t)$. This starts erratically (because the age structure is still adjusting) but converges to the dominant eigenvalue $\lambda_1$. The speed of convergence depends on the ratio $|\lambda_1 / \lambda_2|$ — the larger this ratio, the faster the convergence.

In [ ]:
total_real = np.sum(history_real, axis=0)
years = np.arange(history_real.shape[1]) * 5

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(years, total_real, 'b-o', lw=2, markersize=3)
ax1.set_xlabel('Year')
ax1.set_ylabel('Total Population (thousands)')
ax1.set_title('Total Population Projection', fontweight='bold')
ax1.grid(True, alpha=0.3)

# Growth factor per step
growth_factors = total_real[1:] / total_real[:-1]
ax2.plot(years[1:], growth_factors, 'r-o', lw=2, markersize=3)
ax2.axhline(lam_real, color='gray', ls='--', label=f'$\\lambda_1 = {lam_real:.4f}$')
ax2.set_xlabel('Year')
ax2.set_ylabel('Growth Factor (per 5 years)')
ax2.set_title('Convergence of Growth Factor', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 7 · Your Tasks

You must implement a Leslie matrix model for a **real country** using real demographic data. Additionally, incorporate **at least two** of the following features:

### Task A: Changing Fertility Rates
Introduce time-dependent fertility rates to simulate demographic trends.
- Model declining birth rates (e.g., reduce fertility by 1–2% per decade)
- Show how declining fertility changes the population pyramid shape
- Compare projections with and without fertility decline

### Task B: Stochastic Survival Rates
Add random variability to survival rates.
- Each time step, perturb survival rates by a small random factor
- Run multiple realisations to quantify uncertainty in projections
- Show confidence intervals around the total population trajectory

### Task C: Migration Effects
Include immigration and emigration.
- Add a migration vector $\mathbf{M}$ so that $\mathbf{N}(t+1) = \mathbf{L} \cdot \mathbf{N}(t) + \mathbf{M}$
- Model age-specific migration (e.g., young adults emigrate, families immigrate)
- Analyse how migration changes long-term population structure

### Task D: Multiple Subpopulations
Differentiate between groups (urban vs rural) with distinct demographic parameters.
- Build separate Leslie matrices for each group
- Allow migration between groups
- Compare the dynamics of the combined system

### Data sources
- [UN World Population Prospects](https://population.un.org/wpp/) — fertility, mortality, and population by age
- [World Bank](https://data.worldbank.org/) — demographic indicators
- Czech data available in `czech_population/` directory

In [ ]:
# TODO: Delete this cell

---
## Recommended Reading & Journal Club

### Foundational References

**1. Leslie, P. H. (1945)**
*On the use of matrices in certain population mathematics.*
Biometrika, 33(3), 183–212. [DOI](https://doi.org/10.1093/biomet/33.3.183)
→ The original paper introducing the Leslie matrix model.

**2. Caswell, H. (2001)**
*Matrix Population Models: Construction, Analysis, and Interpretation.* 2nd ed.
Sinauer Associates.
→ The definitive textbook. Covers eigenvalue analysis, sensitivity, and applications.

---

### Journal Club Papers

**3. Vollset, S. E. et al. (2020)**
*Fertility, mortality, migration, and population scenarios for 195 countries and territories from 2017 to 2100.*
The Lancet, 396(10258), 1285–1306. [DOI](https://doi.org/10.1016/S0140-6736(20)30677-2)
→ State-of-the-art population projections predicting a peak around 2064. Uses age-structured models similar to Leslie matrices.

**4. Lee, R. D. & Carter, L. R. (1992)**
*Modeling and forecasting US mortality.*
Journal of the American Statistical Association, 87(419), 659–671. [DOI](https://doi.org/10.1080/01621459.1992.10475265)
→ Classic paper on forecasting age-specific mortality rates. The Lee-Carter method is still widely used.

**5. Keyfitz, N. (1971)**
*On the momentum of population growth.*
Demography, 8(1), 71–80. [DOI](https://doi.org/10.2307/2060339)
→ Explains why populations continue growing even after replacement fertility is reached — a direct consequence of age structure.

**6. Salguero-Gómez, R. et al. (2015)**
*The COMPADRE Plant Matrix Database: an open online repository for plant demography.*
Journal of Ecology, 103(1), 202–218. [DOI](https://doi.org/10.1111/1365-2745.12334)
→ Massive database of matrix population models for plants — can be used for comparative analysis across species.